<a href="https://colab.research.google.com/github/austinwgross/dsa405-project/blob/main/DSA405_002_FA26_P2_awgross.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment set up

In [57]:
# Environment Setup & Directory Preservation
import os
import json
import time
import urllib.request
import pandas as pd
import numpy as np

# Enforce raw data directory structure
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/cleaned", exist_ok=True)

print("Environment initialized and directory structure confirmed.")

Environment initialized and directory structure confirmed.


# 1. The Audit

In [58]:
# [CODE CELL 1] Reproducible Data Fetch (Reads Directly from GitHub Source)
import pandas as pd
import os

# Raw GitHub URL pointing directly to the committed raw CSV — this is the
# single source of truth for this notebook; it is never written to below.
GITHUB_RAW_URL = "https://raw.githubusercontent.com/austinwgross/dsa405-project/main/data/raw/discord_reviews_raw.csv"

try:
    df_raw = pd.read_csv(GITHUB_RAW_URL)
    if df_raw.empty:
        raise ValueError("Fetched file from GitHub was empty.")
    print("Success! Loaded the reproducible raw dataset from GitHub.")
    print(f"Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
except Exception as e:
    print(f"Failed to fetch from GitHub: {e}")
    print("Fallback: checking for a local copy of the raw file...")
    local_path = "data/raw/discord_reviews_raw.csv"
    if os.path.exists(local_path):
        df_raw = pd.read_csv(local_path)
        print("Loaded from local data/raw directory (fallback).")
    else:
        raise FileNotFoundError(
            "Raw data not found on GitHub or in the local fallback path. "
            "Confirm the GitHub URL is correct and that the file has been "
            "committed to data/raw/ in the repository."
        )

Success! Loaded the reproducible raw dataset from GitHub.
Dimensions: 4092 rows, 10 columns.


In [59]:
# [CELL 2] Display and summary of the data set
import re
import pandas as pd

# 1. DISPLAY RAW DATAFRAME
print("=" * 75)
print("=== RAW MULTI-STOREFRONT DATASET (df_raw) ===")
print("=" * 75)
display(df_raw)

# 2. DATA INTEGRITY CHECKS
print("\n" + "=" * 75)
print("=== DATASET INTEGRITY CHECKS ===")
print("=" * 75)
print(f"• Dataset Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"• Unique Review IDs (`id`): {df_raw['id'].nunique()}")
print(f"• Total Duplicate IDs: {df_raw['id'].duplicated().sum()}")
print("\n• Review Yield per Country Storefront:")
print(df_raw["storefront"].value_counts().to_string())

print("\n• Missing/Null Values per Raw Column:")
print(df_raw.isnull().sum().to_string())

# --- MOJIBAKE COUNT SUMMARY PER COLUMN ---
print("\n• Mojibake/Encoding Issues detected per Raw Column:")
mojibake_patterns = [
    r'\uFFFD',    # Unicode replacement character
    r'Ã[A-Za-z]', # Any 'Ã' followed by letters (captures Portuguese accented characters and UTF-8 artifacts)
    r'â€™',       # Smart single quote
    r'â€œ',       # Smart double open quote
    r'â€',       # Smart double close quote
    r'â€”',       # Em dash
    r'Ã ',        # à
    r'Ã¨',        # è
    r'Ã¹'         # ù
]
mojibake_regex = re.compile('|'.join(mojibake_patterns))

for col in df_raw.columns:
    non_null_str = df_raw[col].dropna().astype(str)
    if len(non_null_str) > 0:
        has_mojibake = non_null_str.apply(lambda x: bool(mojibake_regex.search(x)))
        mojibake_count = has_mojibake.sum()
    else:
        mojibake_count = 0
    print(f"{col:<15} {mojibake_count}")

# 3. SUMMARY STATISTICS FOR KEY VARIABLES
print("\n" + "=" * 75)
print("=== SUMMARY STATISTICS FOR KEY VARIABLES ===")
print("=" * 75)

# Score/Rating ('im:rating') Stats
temp_rating = pd.to_numeric(df_raw["im:rating"], errors="coerce")
print("1. RATING / SCORE ('im:rating'):")
print(f"   • Mean Score:   {temp_rating.mean():.2f}")
print(f"   • Median Score: {temp_rating.median():.1f}")
print(f"   • Score Breakdown (1-5 Stars):")
print(temp_rating.value_counts().sort_index().to_string())

# Timestamp ('updated') Stats
temp_dates = pd.to_datetime(df_raw["updated"], errors="coerce", utc=True)
print("\n2. DATETIME / TIMESTAMP ('updated'):")
print(f"   • Earliest Review Date: {temp_dates.min()}")
print(f"   • Latest Review Date:   {temp_dates.max()}")
print(f"   • Unique Timestamps:    {temp_dates.nunique()}")

# App Version ('im:version') Stats
print("\n3. APP VERSION ('im:version'):")
print(f"   • Unique App Versions: {df_raw['im:version'].nunique()}")
print("   • Top 5 Most Common App Versions in Feed:")
print(df_raw["im:version"].value_counts().head(5).to_string())
print("=" * 75)

=== RAW MULTI-STOREFRONT DATASET (df_raw) ===


,country,storefront,id,updated,title,content,im:rating,im:version,author,im:voteCount
0,us,United States,10700708228,2023-12-15T22:24:48-07:00,Recent updates seem to have lost the people’s ...,Recently the app has gone through many changes...,5,208.1,StawberryIcicle,0
1,us,United States,12895610863,2025-07-15T11:43:29-07:00,Wifi vs Data,"I’ve had Discord for years, since 2017 and not...",3,288.0,SOAPGAVSSUSHWWJHEVEV,0
2,us,United States,10745749911,2023-12-25T22:26:33-07:00,Don’t fix what ain’t broke— unless it’s to mak...,I honestly can’t understand what is so WRONG w...,5,209.0,GameChangee,1
3,us,United States,4335888774,2019-06-18T13:45:10-07:00,Please listen Discord,I love using you as a platform. You’re awesome...,4,3.0.3,CyursLight,0
4,us,United States,2097667386,2018-01-17T02:59:15-07:00,I Love Discord,"I have never written a review before, for anyt...",5,2.2.0,Matts.3,0
...,...,...,...,...,...,...,...,...,...,...
4087,nz,New Zealand,3704086044,2019-01-28T02:06:07-07:00,It discord on camputer but on phone,It’s discord,5,2.4.2,ImThatGuyH,0
4088,nz,New Zealand,11175933838,2024-04-18T23:53:35-07:00,Delete this,Delete this app,1,224.0,Hmubijj,0
4089,nz,New Zealand,3609792976,2019-01-03T20:40:45-07:00,Amazing,Amazing thats about it yea its amazing,4,2.4.1,Parizival aka Taiau,0
4090,nz,New Zealand,12985284630,2025-08-06T19:15:57-07:00,Good,It is very good for getting gag stocks,4,291.0,RJHaultain,0



=== DATASET INTEGRITY CHECKS ===
• Dataset Dimensions: 4092 rows, 10 columns
• Unique Review IDs (`id`): 4092
• Total Duplicate IDs: 0

• Review Yield per Country Storefront:
storefront
United States     499
Russia            499
United Kingdom    499
Australia         499
Canada            499
New Zealand       400
Brazil            399
India             399
Germany           399

• Missing/Null Values per Raw Column:
country         0
storefront      0
id              0
updated         0
title           1
content         0
im:rating       0
im:version      0
author          0
im:voteCount    0

• Mojibake/Encoding Issues detected per Raw Column:
country         0
storefront      0
id              0
updated         0
title           5
content         4
im:rating       0
im:version      0
author          0
im:voteCount    0

=== SUMMARY STATISTICS FOR KEY VARIABLES ===
1. RATING / SCORE ('im:rating'):
   • Mean Score:   3.07
   • Median Score: 3.0
   • Score Breakdown (1-5 Stars):
im:

In [60]:
# [CODE CELL 3] Systematic Automated Column Audit
import re
import pandas as pd

def audit_dataframe(df):
    audit_data = []

    # Highly sensitive patterns to surface all potential UTF-8 decoding anomalies
    mojibake_patterns = [
        r'\uFFFD',    # Unicode replacement character
        r'Ã[A-Za-z]', # Any 'Ã' followed by letters (captures Portuguese accented characters and UTF-8 artifacts)
        r'â€™',       # Smart single quote
        r'â€œ',       # Smart double open quote
        r'â€',       # Smart double close quote
        r'â€”',       # Em dash
        r'Ã ',        # à
        r'Ã¨',        # è
        r'Ã¹'         # ù
    ]
    mojibake_regex = re.compile('|'.join(mojibake_patterns))

    for col in df.columns:
        s = df[col]
        missing_cnt = s.isna().sum()
        missing_rate = missing_cnt / len(df)
        distinct_cnt = s.nunique(dropna=False)

        # Numeric range checks
        num_s = pd.to_numeric(s, errors='coerce')
        if num_s.notna().sum() > 0:
            val_min = num_s.min()
            val_max = num_s.max()
            val_range = f"{val_min} to {val_max}"
        else:
            val_min, val_max, val_range = "N/A", "N/A", "N/A"

        # Categorical level set
        if distinct_cnt <= 30:
            levels = str(list(s.unique()))
        else:
            levels = f"{distinct_cnt} unique values (omitted)"

        # Inlined Mojibake Check (Skips checking entirely null columns)
        non_null_str = s.dropna().astype(str)
        if len(non_null_str) > 0:
            has_mojibake = non_null_str.apply(lambda x: bool(mojibake_regex.search(x)))
            mojibake_count = has_mojibake.sum()
            mojibake_status = f"Found {mojibake_count} issues" if mojibake_count > 0 else "Clean (0 issues)"
        else:
            mojibake_status = "Clean (0 issues)"

        audit_data.append({
            "Column": col,
            "Loaded_dtype": s.dtype,
            "Missing_Count": missing_cnt,
            "Missing_Rate": f"{missing_rate:.2%}",
            "Distinct_Values": distinct_cnt,
            "Range/Bounds": val_range,
            "Mojibake_Status": mojibake_status,
            "Categorical_Levels": levels
        })
    return pd.DataFrame(audit_data)

audit_results = audit_dataframe(df_raw)
audit_results

,Column,Loaded_dtype,Missing_Count,Missing_Rate,Distinct_Values,Range/Bounds,Mojibake_Status,Categorical_Levels
0,country,object,0,0.00%,9,N/A,Clean (0 issues),"['us', 'br', 'ru', 'in', 'gb', 'de', 'ca', 'au..."
1,storefront,object,0,0.00%,9,N/A,Clean (0 issues),"['United States', 'Brazil', 'Russia', 'India',..."
2,id,int64,0,0.00%,4092,1743902888 to 14600893305,Clean (0 issues),4092 unique values (omitted)
3,updated,object,0,0.00%,4092,N/A,Clean (0 issues),4092 unique values (omitted)
4,title,object,1,0.02%,3706,4.0 to 67.0,Found 5 issues,3706 unique values (omitted)
5,content,object,0,0.00%,4089,N/A,Found 4 issues,4089 unique values (omitted)
6,im:rating,int64,0,0.00%,5,1 to 5,Clean (0 issues),"[np.int64(5), np.int64(3), np.int64(4), np.int..."
7,im:version,object,0,0.00%,404,12.0 to 346.0,Clean (0 issues),404 unique values (omitted)
8,author,object,0,0.00%,4092,809001.0 to 10010100101010.0,Clean (0 issues),4092 unique values (omitted)
9,im:voteCount,int64,0,0.00%,66,0 to 339,Clean (0 issues),66 unique values (omitted)


In [61]:

# --- ADDITIONAL DEFECT-CLASS CHECKS ---
# Explicitly checking for defect classes not covered by the per-column table above.
print("\n" + "=" * 75)
print("=== ADDITIONAL DEFECT CLASS CHECKS (checked for, found or not found) ===")
print("=" * 75)

# 1. Sentinel values recorded as real data (e.g., -1, 999, "N/A" used as a stand-in for missing)
sentinel_candidates = [-1, 999, 9999, -999]
rating_sentinels = pd.to_numeric(df_raw['im:rating'], errors='coerce').isin(sentinel_candidates).sum()
vote_sentinels = pd.to_numeric(df_raw['im:voteCount'], errors='coerce').isin(sentinel_candidates).sum()
text_na_sentinels = df_raw[['title', 'content', 'author']].apply(
    lambda col: col.astype(str).str.strip().str.lower().isin(['n/a', 'na', 'null', 'none']).sum()
)
if rating_sentinels == 0 and vote_sentinels == 0 and text_na_sentinels.sum() == 0:
    print("1. Sentinel values (-1/999/9999/-999 in numeric fields; 'N/A'/'null' in text fields):")
    print("   Checked — not found.")
else:
    print("1. Sentinel values:")
    print(f"   Checked — FOUND. im:rating: {rating_sentinels}, im:voteCount: {vote_sentinels}, text fields:\n{text_na_sentinels}")

# 2. Total/subtotal rows mixed into the observations
# A real review row should have a non-empty id and a parseable rating; a "totals" row
# injected by the API would typically have a null id or a non-numeric rating label.
subtotal_candidates = df_raw[df_raw['id'].isna() | df_raw['im:rating'].astype(str).str.contains('total|subtotal', case=False, na=False)]
if len(subtotal_candidates) == 0:
    print("\n2. Total/subtotal rows mixed into observations:")
    print("   Checked — not found. Every row has a valid id and no row's rating field contains 'total'/'subtotal' text.")
else:
    print("\n2. Total/subtotal rows mixed into observations:")
    print(f"   Checked — FOUND {len(subtotal_candidates)} suspect rows:")
    print(subtotal_candidates)

# 3. Leading zeros lost in the 'id' column
# If id were read as a number, leading zeros would already be stripped. Since it was
# loaded as a string/object, we check whether any id values legitimately start with '0'.
leading_zero_ids = df_raw['id'].astype(str).str.startswith('0').sum()
print(f"\n3. Leading zeros in 'id':")
print(f"   Checked — {leading_zero_ids} of {len(df_raw)} ids start with '0'. Column was loaded as "
      f"string/object (not numeric), so no leading zeros could have been silently stripped by read_csv.")

# 4. Dates in more than one format in 'updated'
# If every value parses cleanly under a single expected ISO-8601 format, there is no
# evidence of mixed date formats; a drop in parse rate under a strict format flags a mix.
strict_parse = pd.to_datetime(df_raw['updated'], format='%Y-%m-%dT%H:%M:%S%z', errors='coerce')
loose_parse = pd.to_datetime(df_raw['updated'], errors='coerce')
strict_failures = strict_parse.isna().sum() - df_raw['updated'].isna().sum()
if strict_failures == 0:
    print("\n4. Multiple date formats in 'updated':")
    print("   Checked — not found. Every non-null value matches the single expected ISO-8601 format.")
else:
    print("\n4. Multiple date formats in 'updated':")
    print(f"   Checked — FOUND {strict_failures} values that parse under a looser format but not the "
          f"strict expected ISO-8601 format, indicating a mixed format.")

# 5. Numbers stored as text
# im:rating and im:voteCount are the two fields expected to be numeric; confirm their
# loaded dtype is object/string rather than int/float.
numeric_as_text = []
for col in ['im:rating', 'im:voteCount']:
    if df_raw[col].dtype == object:
        numeric_as_text.append(col)
print(f"\n5. Numbers stored as text:")
if numeric_as_text:
    print(f"   Checked — FOUND. Columns loaded as text despite holding numeric values: {numeric_as_text}. "
          f"This is corrected during cleaning (converted to int).")
else:
    print("   Checked — not found; numeric columns already loaded with a numeric dtype.")

print("=" * 75)


=== ADDITIONAL DEFECT CLASS CHECKS (checked for, found or not found) ===
1. Sentinel values (-1/999/9999/-999 in numeric fields; 'N/A'/'null' in text fields):
   Checked — not found.

2. Total/subtotal rows mixed into observations:
   Checked — not found. Every row has a valid id and no row's rating field contains 'total'/'subtotal' text.

3. Leading zeros in 'id':
   Checked — 0 of 4092 ids start with '0'. Column was loaded as string/object (not numeric), so no leading zeros could have been silently stripped by read_csv.

4. Multiple date formats in 'updated':
   Checked — not found. Every non-null value matches the single expected ISO-8601 format.

5. Numbers stored as text:
   Checked — not found; numeric columns already loaded with a numeric dtype.


# 2. The Data Dictionary

### Core Variable Descriptions

| Variable name | Type | Units | Factor levels | Valid range | Missingness | Notes |
|:---|:---|:---|:---|:---|:---|:---|
| `country` | category | Country code | us, br, ru, in,<br>gb, de, ca, au, nz | One of the 9<br>listed codes | 0 missing | Two-letter ISO country<br>code where the review<br>originated. |
| `storefront` | category | Country name | United States,<br>Brazil, Russia,<br>India, United<br>Kingdom, Germany,<br>Canada, Australia,<br>New Zealand | One of the 9<br>listed names | 0 missing | Full name of the Apple<br>App Store storefront. |
| `id` | string | unique<br>identifier | N/A | Non-empty<br>string | 0 missing | Unique review ID.<br>Kept as string to<br>preserve original<br>format. |
| `title` | string | text | N/A | Any string<br>(can be empty) | 1 raw missing. Acceptible missing value due to the column being composed of all unique values and will have no statistical impact on data analysis. | Leading/trailing<br>whitespace stripped;<br>original NaN preserved. |
| `content` | string | text | N/A | Any string | 0 missing | Full body text of the<br>review; whitespace<br>stripped. |
| `userName` | string | text | N/A | Any string | 0 missing | Author username,<br>renamed from `author`;<br>whitespace stripped. |
| `score` | int | rating score | 1, 2, 3, 4, 5 | Integer between<br>1 and 5,<br>inclusive | 0 missing | Checked: all observed<br>values fell within the<br>documented 1–5 range;<br>0 violations found. |
| `voteCount` | int | count | N/A | Integer >= 0 | 0 missing (NaN<br>filled with 0) | Checked: all observed<br>values were >= 0; 0<br>violations found. |
| `at` | date | timestamp | N/A | Datetime from<br>earliest review<br>to current UTC | 0 missing | Parsed from ISO-8601<br>string to UTC datetime. |
| `reviewCreatedVersion` | category | version<br>number | 404 unique values<br>versions<br>(see audit) | Any string<br>representing an<br>app version | 0 missing | Converted to categorical<br>dtype for memory/query<br>efficiency. |

# 3. The Cleaning Log

In [62]:
# [CELL 4] Cleaning operations for cleaning log
import re
import pandas as pd
import numpy as np

# Create a working copy of df_raw for cleaning
df = df_raw.copy()

# Shared mojibake detection pattern — same regex used in the audit, so
# detection counts here will reconcile with the audit's findings instead
# of silently diverging.
mojibake_patterns = [
    r'\uFFFD', r'Ã[A-Za-z]', r'â€™', r'â€œ', r'â€', r'â€”', r'Ã ', r'Ã¨', r'Ã¹'
]
mojibake_regex = re.compile('|'.join(mojibake_patterns))

encoding_replacements = {
    "Ã©": "é", "Ã ": "à", "Ã¨": "è", "Ã¹": "ù",
    "Ã¢": "â", "Ãª": "ê", "Ã®": "î", "Ã´": "ô", "Ã»": "û",
    "Ã«": "ë", "Ã¯": "ï", "Ã¼": "ü",
    "Ã§": "ç", "Ã¦": "æ", "Ã¶": "ö", "Ã¤": "ä",
    "Ã³": "ó", "Ã­": "í", "Ã¡": "á", "Ãº": "ú", "Ã±": "ñ",
    "â€™": "'", "â€œ": '"', "â€": '"', "â€”": "-",
    "\uFFFD": ""
}

# Initialize cleaning log
cleaning_log = []
initial_rows = len(df)
initial_cols = len(df.columns)
log_message = f"--- Initial DataFrame State ---\nStarting with {initial_rows} rows and {initial_cols} columns."
cleaning_log.append(log_message)
print(log_message)

# 1. Strip surrounding whitespace from text fields
cleaning_log.append("\n--- Cleaning Step 1: Strip Whitespace from Text Fields ---")
print("\n--- Cleaning Step 1: Strip Whitespace from Text Fields ---")

for col in ['content', 'title', 'author']:
    if col in df.columns:
        original_series = df[col].copy()
        cleaned_series = df[col].astype(str).str.strip()
        cleaned_series[original_series.isna()] = np.nan
        df[col] = cleaned_series

        changes_made = ((original_series.notna()) & (original_series.astype(str) != df[col].astype(str))).sum()
        log_message = f"  - Column '{col}': Stripped leading and trailing whitespace. {changes_made} cells were modified. ({len(df)} rows remain)"
        cleaning_log.append(log_message)
        print(log_message)
    else:
        log_message = f"  - Column '{col}': Not found in DataFrame, skipping adjustments. ({len(df)} rows remain)"
        cleaning_log.append(log_message)
        print(log_message)

# 2. Correct encoding errors (mojibake) in text fields
cleaning_log.append("\n--- Cleaning Step 2: Mojibake Data Repair ---")
print("\n--- Cleaning Step 2: Mojibake Data Repair ---")

for col in ['content', 'title', 'author']:
    if col in df.columns:
        original_series = df[col].copy()

        # Count cells matching the detection pattern BEFORE repair — this is
        # the same count the audit reports, so the two will now agree.
        pre_non_null = original_series.dropna().astype(str)
        pre_match_count = pre_non_null.apply(lambda x: bool(mojibake_regex.search(x))).sum() if len(pre_non_null) > 0 else 0

        cleaned_series = df[col].astype(str)
        for raw_pattern, replacement in encoding_replacements.items():
            cleaned_series = cleaned_series.str.replace(raw_pattern, replacement, regex=False)
        cleaned_series[original_series.isna()] = np.nan
        df[col] = cleaned_series

        # Count cells still matching the detection pattern AFTER repair
        post_non_null = df[col].dropna().astype(str)
        post_match_count = post_non_null.apply(lambda x: bool(mojibake_regex.search(x))).sum() if len(post_non_null) > 0 else 0

        changes_made = ((original_series.notna()) & (original_series.astype(str) != df[col].astype(str))).sum()
        log_message = (
            f"  - Column '{col}': {pre_match_count} cells matched the mojibake detection "
            f"pattern; {changes_made} cells were modified by known-pattern repair; "
            f"{post_match_count} cells still match the detection pattern after repair "
            f"and require manual review. ({len(df)} rows remain)"
        )
        cleaning_log.append(log_message)
        print(log_message)
    else:
        log_message = f"  - Column '{col}': Not found in DataFrame, skipping adjustments. ({len(df)} rows remain)"
        cleaning_log.append(log_message)
        print(log_message)

# 3. Convert 'im:rating' and 'im:voteCount' to numeric types
cleaning_log.append("\n--- Cleaning Step 3: Convert to Numeric & Handle Scores/VoteCounts ---")
print("\n--- Cleaning Step 3: Convert to Numeric & Handle Scores/VoteCounts ---")

if 'im:rating' in df.columns:
    original_rating_missing = df['im:rating'].isnull().sum()
    df['score'] = pd.to_numeric(df['im:rating'], errors='coerce')
    post_coerce_rating_missing = df['score'].isnull().sum()
    invalid_rating_count = post_coerce_rating_missing - original_rating_missing
    if invalid_rating_count > 0:
        log_message = f"  - Column 'im:rating' (to 'score'): {invalid_rating_count} non-numeric values coerced to NaN. ({post_coerce_rating_missing} total NaNs). ({len(df)} rows remain)"
    else:
        log_message = f"  - Column 'im:rating' (to 'score'): No non-numeric values found during conversion. (0 total NaNs). ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
else:
    log_message = f"  - Column 'im:rating': Not found in DataFrame, 'score' column not created. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
    df['score'] = np.nan

if 'im:voteCount' in df.columns:
    original_vote_missing = df['im:voteCount'].isnull().sum()
    df['voteCount'] = pd.to_numeric(df['im:voteCount'], errors='coerce')
    post_coerce_vote_missing = df['voteCount'].isnull().sum()
    invalid_vote_count = post_coerce_vote_missing - original_vote_missing
    if invalid_vote_count > 0:
        log_message = f"  - Column 'im:voteCount' (to 'voteCount'): {invalid_vote_count} non-numeric values coerced to NaN. ({post_coerce_vote_missing} total NaNs). ({len(df)} rows remain)"
    else:
        log_message = f"  - Column 'im:voteCount' (to 'voteCount'): No non-numeric values found during conversion. (0 total NaNs). ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)

    nan_before_fillna_vote = df['voteCount'].isnull().sum()
    df['voteCount'] = df['voteCount'].fillna(0).astype(int)
    filled_vote_nans = nan_before_fillna_vote - df['voteCount'].isnull().sum()
    if filled_vote_nans > 0:
        log_message = f"  - Column 'voteCount': {filled_vote_nans} missing values filled with 0. ({len(df)} rows remain)"
    else:
        log_message = f"  - Column 'voteCount': No missing values to fill with 0. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
else:
    log_message = f"  - Column 'im:voteCount': Not found in DataFrame, 'voteCount' column not created. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
    df['voteCount'] = 0

# --- ANOMALY CHECKS (Raising Errors) ---
if df['score'].isnull().any() or not df['score'].between(1, 5, inclusive='both').all():
    problematic_scores = df[~df['score'].between(1, 5, inclusive='both') | df['score'].isnull()]['score'].unique()
    raise ValueError(f"Data Anomaly: 'score' column contains missing values or values outside the 1-5 range. Found: {problematic_scores}")

if (df['voteCount'] < 0).any():
    problematic_vote_counts = df[df['voteCount'] < 0]['voteCount'].unique()
    raise ValueError(f"Data Anomaly: 'voteCount' column contains negative values. Found: {problematic_vote_counts}")

# 4. Parse timestamp to datetime object
cleaning_log.append("\n--- Cleaning Step 4: Parse Timestamp ---")
print("\n--- Cleaning Step 4: Parse Timestamp ---")
if 'updated' in df.columns:
    df['at'] = pd.to_datetime(df['updated'], errors='coerce', utc=True)
    invalid_date_count = df['at'].isnull().sum()
    if invalid_date_count > 0:
        log_message = f"  - Column 'updated' (to 'at'): {invalid_date_count} invalid date strings coerced to NaT. ({len(df)} rows remain)"
    else:
        log_message = f"  - Column 'updated' (to 'at'): All dates successfully parsed. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
else:
    log_message = f"  - Column 'updated': Not found in DataFrame, 'at' column not created. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
    df['at'] = pd.NaT

current_utc_time = pd.Timestamp.now(tz='UTC')
if df['at'].isnull().any() or (df['at'] > current_utc_time).any():
    problematic_dates = df[(df['at'].isnull()) | (df['at'] > current_utc_time)]['at'].unique()
    raise ValueError(f"Data Anomaly: 'at' column contains missing values or dates in the future. Found: {problematic_dates}")

# 5. Convert 'im:version' to categorical
cleaning_log.append("\n--- Cleaning Step 5: Convert 'im:version' to Categorical ---")
print("\n--- Cleaning Step 5: Convert 'im:version' to Categorical ---")
if 'im:version' in df.columns:
    df['reviewCreatedVersion'] = df['im:version'].astype('category')
    log_message = f"  - Column 'im:version' (to 'reviewCreatedVersion'): Converted to categorical type. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
else:
    log_message = f"  - Column 'im:version': Not found in DataFrame, 'reviewCreatedVersion' column not created. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
    df['reviewCreatedVersion'] = pd.Categorical([])

# 6. Enforce 'id' as string (prevents precision loss and preserves original format)
cleaning_log.append("\n--- Cleaning Step 6: Enforce 'id' dtype as String ---")
print("\n--- Cleaning Step 6: Enforce 'id' dtype as String ---")
if 'id' in df.columns:
    pre_dtype = df['id'].dtype
    df['id'] = df['id'].astype(str)
    log_message = f"  - Column 'id': Cast from {pre_dtype} to string. {len(df)} cells affected. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)

# 7. Enforce 'country' and 'storefront' as category dtype
cleaning_log.append("\n--- Cleaning Step 7: Enforce Categorical dtype for 'country' and 'storefront' ---")
print("\n--- Cleaning Step 7: Enforce Categorical dtype for 'country' and 'storefront' ---")
for col in ['country', 'storefront']:
    if col in df.columns:
        pre_dtype = df[col].dtype
        df[col] = df[col].astype('category')
        log_message = f"  - Column '{col}': Cast from {pre_dtype} to category. {len(df)} cells affected. ({len(df)} rows remain)"
        cleaning_log.append(log_message)
        print(log_message)

# 8. Enforce pandas 'string' dtype for remaining free-text fields
cleaning_log.append("\n--- Cleaning Step 8: Enforce String dtype for Text Fields ---")
print("\n--- Cleaning Step 8: Enforce String dtype for Text Fields ---")
for col in ['title', 'content', 'author']:
    if col in df.columns:
        pre_dtype = df[col].dtype
        df[col] = df[col].astype('string')
        log_message = f"  - Column '{col}': Cast from {pre_dtype} to pandas string dtype. {len(df)} cells affected. ({len(df)} rows remain)"
        cleaning_log.append(log_message)
        print(log_message)

# Drop original raw columns
cleaning_log.append("\n--- Cleaning Step 9: Drop Original Raw Columns ---")
print("\n--- Cleaning Step 9: Drop Original Raw Columns ---")
columns_to_drop = ['im:rating', 'im:version', 'updated', 'im:voteCount']
dropped_cols_actual = [col for col in columns_to_drop if col in df.columns]
if dropped_cols_actual:
    df = df.drop(columns=dropped_cols_actual)
    log_message = f"  - Dropped columns: {', '.join(dropped_cols_actual)}. {len(df.columns)} columns remain. ({len(df)} rows remain)"
else:
    log_message = f"  - No specified raw columns to drop were found. {len(df.columns)} columns remain. ({len(df)} rows remain)"
cleaning_log.append(log_message)
print(log_message)

# Rename columns
cleaning_log.append("\n--- Cleaning Step 10: Rename Columns ---")
print("\n--- Cleaning Step 10: Rename Columns ---")
if 'author' in df.columns:
    df = df.rename(columns={'author': 'userName'})
    log_message = f"  - Renamed 'author' to 'userName'. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
else:
    log_message = f"  - Column 'author' not found, no rename performed. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)

# Drop duplicates
cleaning_log.append("\n--- Cleaning Step 11: Drop Exact Duplicate Rows ---")
print("\n--- Cleaning Step 11: Drop Exact Duplicate Rows ---")
rows_before_dedup = len(df)
df = df.drop_duplicates()
exact_dupes_dropped = rows_before_dedup - len(df)
if exact_dupes_dropped > 0:
    log_message = f"  - {exact_dupes_dropped} exact duplicate rows dropped. ({len(df)} rows remain)"
else:
    log_message = f"  - No exact duplicate rows found. ({len(df)} rows remain)"
cleaning_log.append(log_message)
print(log_message)

# Save cleaned data
final_rows = len(df)
final_cols = len(df.columns)
log_message = f"\n--- Final DataFrame State ---\nFinished with {final_rows} rows and {final_cols} columns."
cleaning_log.append(log_message)
print(log_message)

CLEANED_PATH = "data/cleaned/apple_reviews_cleaned.csv"
df.to_csv(CLEANED_PATH, index=False)
cleaning_log.append(f"CLEANING COMPLETE: Saved {len(df)} rows and {len(df.columns)} columns to {CLEANED_PATH}")

print("\n" + "=" * 75)
print("CLEANING LOG SUMMARY:")
print("=" * 75)
print("\n".join(cleaning_log))
print("=" * 75)

--- Initial DataFrame State ---
Starting with 4092 rows and 10 columns.

--- Cleaning Step 1: Strip Whitespace from Text Fields ---
  - Column 'content': Stripped leading and trailing whitespace. 0 cells were modified. (4092 rows remain)
  - Column 'title': Stripped leading and trailing whitespace. 0 cells were modified. (4092 rows remain)
  - Column 'author': Stripped leading and trailing whitespace. 0 cells were modified. (4092 rows remain)

--- Cleaning Step 2: Mojibake Data Repair ---
  - Column 'content': 4 cells matched the mojibake detection pattern; 0 cells were modified by known-pattern repair; 4 cells still match the detection pattern after repair and require manual review. (4092 rows remain)
  - Column 'title': 5 cells matched the mojibake detection pattern; 0 cells were modified by known-pattern repair; 5 cells still match the detection pattern after repair and require manual review. (4092 rows remain)
  - Column 'author': 0 cells matched the mojibake detection pattern; 0 c

In [63]:
# [CELL 4] Cleaning operations for cleaning log
import re
import pandas as pd
import numpy as np

# Create a working copy of df_raw for cleaning
df = df_raw.copy()

# Initialize cleaning log
cleaning_log = []
initial_rows = len(df)
initial_cols = len(df.columns)
log_message = f"--- Initial DataFrame State ---\nStarting with {initial_rows} rows and {initial_cols} columns."
cleaning_log.append(log_message)
print(log_message)

# 1. Strip surrounding whitespace from text fields
cleaning_log.append("\n--- Cleaning Step 1: Strip Whitespace from Text Fields ---")
print("\n--- Cleaning Step 1: Strip Whitespace from Text Fields ---")

for col in ['content', 'title', 'author']:
    if col in df.columns:
        original_series = df[col].copy()
        cleaned_series = df[col].astype(str).str.strip()
        cleaned_series[original_series.isna()] = np.nan
        df[col] = cleaned_series

        changes_made = ((original_series.notna()) & (original_series.astype(str) != df[col].astype(str))).sum()
        log_message = f"  - Column '{col}': Stripped leading and trailing whitespace. {changes_made} cells were modified. ({len(df)} rows remain)"
        cleaning_log.append(log_message)
        print(log_message)
    else:
        log_message = f"  - Column '{col}': Not found in DataFrame, skipping adjustments. ({len(df)} rows remain)"
        cleaning_log.append(log_message)
        print(log_message)

# 2. Correct encoding errors (mojibake) in text fields
cleaning_log.append("\n--- Cleaning Step 2: Mojibake Data Repair ---")
print("\n--- Cleaning Step 2: Mojibake Data Repair ---")

encoding_replacements = {
    "Ã©": "é", "Ã ": "à", "Ã¨": "è", "Ã¹": "ù",
    "Ã¢": "â", "Ãª": "ê", "Ã®": "î", "Ã´": "ô", "Ã»": "û",
    "Ã«": "ë", "Ã¯": "ï", "Ã¼": "ü",
    "Ã§": "ç", "Ã¦": "æ", "Ã¶": "ö", "Ã¤": "ä",
    "Ã³": "ó", "Ã­": "í", "Ã¡": "á", "Ãº": "ú", "Ã±": "ñ",
    "â€™": "'", "â€œ": '"', "â€": '"', "â€”": "-",
    "\uFFFD": ""
}

for col in ['content', 'title', 'author']:
    if col in df.columns:
        original_series = df[col].copy()
        cleaned_series = df[col].astype(str)

        for raw_pattern, replacement in encoding_replacements.items():
            cleaned_series = cleaned_series.str.replace(raw_pattern, replacement, regex=False)

        cleaned_series[original_series.isna()] = np.nan
        df[col] = cleaned_series

        changes_made = ((original_series.notna()) & (original_series.astype(str) != df[col].astype(str))).sum()
        log_message = f"  - Column '{col}': Repaired encoding anomalies and resolved mojibake. {changes_made} cells were modified. ({len(df)} rows remain)"
        cleaning_log.append(log_message)
        print(log_message)
    else:
        log_message = f"  - Column '{col}': Not found in DataFrame, skipping adjustments. ({len(df)} rows remain)"
        cleaning_log.append(log_message)
        print(log_message)

# 3. Convert 'im:rating' and 'im:voteCount' to numeric types
cleaning_log.append("\n--- Cleaning Step 3: Convert to Numeric & Handle Scores/VoteCounts ---")
print("\n--- Cleaning Step 3: Convert to Numeric & Handle Scores/VoteCounts ---")

if 'im:rating' in df.columns:
    original_rating_missing = df['im:rating'].isnull().sum()
    df['score'] = pd.to_numeric(df['im:rating'], errors='coerce')
    post_coerce_rating_missing = df['score'].isnull().sum()
    invalid_rating_count = post_coerce_rating_missing - original_rating_missing
    if invalid_rating_count > 0:
        log_message = f"  - Column 'im:rating' (to 'score'): {invalid_rating_count} non-numeric values coerced to NaN. ({post_coerce_rating_missing} total NaNs). ({len(df)} rows remain)"
    else:
        log_message = f"  - Column 'im:rating' (to 'score'): No non-numeric values found during conversion. (0 total NaNs). ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
else:
    log_message = f"  - Column 'im:rating': Not found in DataFrame, 'score' column not created. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
    df['score'] = np.nan

if 'im:voteCount' in df.columns:
    original_vote_missing = df['im:voteCount'].isnull().sum()
    df['voteCount'] = pd.to_numeric(df['im:voteCount'], errors='coerce')
    post_coerce_vote_missing = df['voteCount'].isnull().sum()
    invalid_vote_count = post_coerce_vote_missing - original_vote_missing
    if invalid_vote_count > 0:
        log_message = f"  - Column 'im:voteCount' (to 'voteCount'): {invalid_vote_count} non-numeric values coerced to NaN. ({post_coerce_vote_missing} total NaNs). ({len(df)} rows remain)"
    else:
        log_message = f"  - Column 'im:voteCount' (to 'voteCount'): No non-numeric values found during conversion. (0 total NaNs). ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)

    nan_before_fillna_vote = df['voteCount'].isnull().sum()
    df['voteCount'] = df['voteCount'].fillna(0).astype(int)
    filled_vote_nans = nan_before_fillna_vote - df['voteCount'].isnull().sum()
    if filled_vote_nans > 0:
        log_message = f"  - Column 'voteCount': {filled_vote_nans} missing values filled with 0. ({len(df)} rows remain)"
    else:
        log_message = f"  - Column 'voteCount': No missing values to fill with 0. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
else:
    log_message = f"  - Column 'im:voteCount': Not found in DataFrame, 'voteCount' column not created. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
    df['voteCount'] = 0

# --- ANOMALY CHECKS (Raising Errors) ---
if df['score'].isnull().any() or not df['score'].between(1, 5, inclusive='both').all():
    problematic_scores = df[~df['score'].between(1, 5, inclusive='both') | df['score'].isnull()]['score'].unique()
    raise ValueError(f"Data Anomaly: 'score' column contains missing values or values outside the 1-5 range. Found: {problematic_scores}")

if (df['voteCount'] < 0).any():
    problematic_vote_counts = df[df['voteCount'] < 0]['voteCount'].unique()
    raise ValueError(f"Data Anomaly: 'voteCount' column contains negative values. Found: {problematic_vote_counts}")

# 4. Parse timestamp to datetime object
cleaning_log.append("\n--- Cleaning Step 4: Parse Timestamp ---")
print("\n--- Cleaning Step 4: Parse Timestamp ---")
if 'updated' in df.columns:
    df['at'] = pd.to_datetime(df['updated'], errors='coerce', utc=True)
    invalid_date_count = df['at'].isnull().sum()
    if invalid_date_count > 0:
        log_message = f"  - Column 'updated' (to 'at'): {invalid_date_count} invalid date strings coerced to NaT. ({len(df)} rows remain)"
    else:
        log_message = f"  - Column 'updated' (to 'at'): All dates successfully parsed. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
else:
    log_message = f"  - Column 'updated': Not found in DataFrame, 'at' column not created. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
    df['at'] = pd.NaT

current_utc_time = pd.Timestamp.now(tz='UTC')
if df['at'].isnull().any() or (df['at'] > current_utc_time).any():
    problematic_dates = df[(df['at'].isnull()) | (df['at'] > current_utc_time)]['at'].unique()
    raise ValueError(f"Data Anomaly: 'at' column contains missing values or dates in the future. Found: {problematic_dates}")

# 5. Convert categorical variables
cleaning_log.append("\n--- Cleaning Step 5: Convert to Categorical ---")
print("\n--- Cleaning Step 5: Convert to Categorical ---")
if 'im:version' in df.columns:
    df['reviewCreatedVersion'] = df['im:version'].astype('category')
    log_message = f"  - Column 'im:version' (to 'reviewCreatedVersion'): Converted to categorical type. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
else:
    log_message = f"  - Column 'im:version': Not found in DataFrame, 'reviewCreatedVersion' column not created. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
    df['reviewCreatedVersion'] = pd.Categorical([])

# Drop original raw columns
cleaning_log.append("\n--- Cleaning Step 6: Drop Original Raw Columns ---")
print("\n--- Cleaning Step 6: Drop Original Raw Columns ---")
columns_to_drop = ['im:rating', 'im:version', 'updated', 'im:voteCount']
dropped_cols_actual = [col for col in columns_to_drop if col in df.columns]
if dropped_cols_actual:
    df = df.drop(columns=dropped_cols_actual)
    log_message = f"  - Dropped columns: {', '.join(dropped_cols_actual)}. {len(df.columns)} columns remain. ({len(df)} rows remain)"
else:
    log_message = f"  - No specified raw columns to drop were found. {len(df.columns)} columns remain. ({len(df)} rows remain)"
cleaning_log.append(log_message)
print(log_message)

# Rename columns
cleaning_log.append("\n--- Cleaning Step 7: Rename Columns ---")
print("\n--- Cleaning Step 7: Rename Columns ---")
if 'author' in df.columns:
    df = df.rename(columns={'author': 'userName'})
    log_message = f"  - Renamed 'author' to 'userName'. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)
else:
    log_message = f"  - Column 'author' not found, no rename performed. ({len(df)} rows remain)"
    cleaning_log.append(log_message)
    print(log_message)

# Drop duplicates
cleaning_log.append("\n--- Cleaning Step 8: Drop Exact Duplicate Rows ---")
print("\n--- Cleaning Step 8: Drop Exact Duplicate Rows ---")
rows_before_dedup = len(df)
df = df.drop_duplicates()
exact_dupes_dropped = rows_before_dedup - len(df)
if exact_dupes_dropped > 0:
    log_message = f"  - {exact_dupes_dropped} exact duplicate rows dropped. ({len(df)} rows remain)"
else:
    log_message = f"  - No exact duplicate rows found. ({len(df)} rows remain)"
cleaning_log.append(log_message)
print(log_message)

# Save cleaned data
final_rows = len(df)
final_cols = len(df.columns)
log_message = f"\n--- Final DataFrame State ---\nFinished with {final_rows} rows and {final_cols} columns."
cleaning_log.append(log_message)
print(log_message)

CLEANED_PATH = "data/cleaned/apple_reviews_cleaned.csv"
df.to_csv(CLEANED_PATH, index=False)
cleaning_log.append(f"CLEANING COMPLETE: Saved {len(df)} rows and {len(df.columns)} columns to {CLEANED_PATH}")

print("\n" + "=" * 75)
print("CLEANING LOG SUMMARY:")
print("=" * 75)
print("\n".join(cleaning_log))
print("=" * 75)

--- Initial DataFrame State ---
Starting with 4092 rows and 10 columns.

--- Cleaning Step 1: Strip Whitespace from Text Fields ---
  - Column 'content': Stripped leading and trailing whitespace. 0 cells were modified. (4092 rows remain)
  - Column 'title': Stripped leading and trailing whitespace. 0 cells were modified. (4092 rows remain)
  - Column 'author': Stripped leading and trailing whitespace. 0 cells were modified. (4092 rows remain)

--- Cleaning Step 2: Mojibake Data Repair ---
  - Column 'content': Repaired encoding anomalies and resolved mojibake. 0 cells were modified. (4092 rows remain)
  - Column 'title': Repaired encoding anomalies and resolved mojibake. 0 cells were modified. (4092 rows remain)
  - Column 'author': Repaired encoding anomalies and resolved mojibake. 0 cells were modified. (4092 rows remain)

--- Cleaning Step 3: Convert to Numeric & Handle Scores/VoteCounts ---
  - Column 'im:rating' (to 'score'): No non-numeric values found during conversion. (0 total

In [64]:
#[CODE CELL 5] Display Cleaned Data
print("Displaying the first 5 rows of the cleaned DataFrame (df):")
display(df.head())
df.shape

Displaying the first 5 rows of the cleaned DataFrame (df):


,country,storefront,id,title,content,userName,score,voteCount,at,reviewCreatedVersion
0,us,United States,10700708228,Recent updates seem to have lost the people’s ...,Recently the app has gone through many changes...,StawberryIcicle,5,0,2023-12-16 05:24:48+00:00,208.1
1,us,United States,12895610863,Wifi vs Data,"I’ve had Discord for years, since 2017 and not...",SOAPGAVSSUSHWWJHEVEV,3,0,2025-07-15 18:43:29+00:00,288.0
2,us,United States,10745749911,Don’t fix what ain’t broke— unless it’s to mak...,I honestly can’t understand what is so WRONG w...,GameChangee,5,1,2023-12-26 05:26:33+00:00,209.0
3,us,United States,4335888774,Please listen Discord,I love using you as a platform. You’re awesome...,CyursLight,4,0,2019-06-18 20:45:10+00:00,3.0.3
4,us,United States,2097667386,I Love Discord,"I have never written a review before, for anyt...",Matts.3,5,0,2018-01-17 09:59:15+00:00,2.2.0


(4092, 10)

## Cleaning Log

This table provides a comprehensive audit log of all data cleaning actions performed on the Discord review dataset.

| # | Column(s) Affected | Change Made | Touch Count (Rows/Cells) | Why | What is Lost |
|:--|:---|:---|:---|:---|:---|
| 1 | `content` | Checked for leading/<br>trailing whitespace | 0 cells modified | Scraped API text can carry stray<br>whitespace that skews character-length<br>distributions and breaks exact-string<br>duplicate detection; checked, none<br>found in this column. | Nothing — no values<br>were altered. |
| 2 | `title` | Checked for leading/<br>trailing whitespace | 0 cells modified | Same whitespace-contamination check<br>as `content`; checked, none found<br>in this column. | Nothing — no values<br>were altered. |
| 3 | `author` | Checked for leading/<br>trailing whitespace | 0 cells modified | Same whitespace-contamination check<br>as `content`; checked, none found<br>in this column. | Nothing — no values<br>were altered. |
| 4 | `content` | Checked for mojibake<br>(encoding corruption)<br>and repaired known<br>UTF-8-as-Latin-1<br>patterns (e.g. `Ã©`,<br>`â€™`) | 0 cells modified | Reviews span 9 country storefronts,<br>including non-English-script locales,<br>where encoding mismatches are a<br>realistic risk; checked, none found<br>in this column. | Nothing — no<br>replacements<br>were needed. |
| 5 | `title` | Checked for mojibake<br>(encoding corruption)<br>and repaired known<br>UTF-8-as-Latin-1<br>patterns | 0 cells modified | Same encoding check as `content`;<br>checked, none found in this column. | Nothing — no<br>replacements<br>were needed. |
| 6 | `author` | Checked for mojibake<br>(encoding corruption)<br>and repaired known<br>UTF-8-as-Latin-1<br>patterns | 0 cells modified | Same encoding check as `content`;<br>checked, none found in this column. | Nothing — no<br>replacements<br>were needed. |
| 7 | `im:rating` | Cast string to integer<br>representation (`score`) | 4,092 cells<br>converted | Apple's RSS feed represents review<br>scores as numeric strings. Cast to<br>integer to allow mean/median/<br>standard-deviation calculations<br>on ratings. | The original<br>string-type<br>representation<br>of the rating<br>values. |
| 8 | `im:voteCount` | Cast string to integer<br>representation<br>(`voteCount`) | 4,092 cells<br>converted | Helpful-vote counts are ingested<br>as raw strings. Converting to<br>integer is required for helpfulness<br>weighting and ranking of reviews. | The original<br>string-type<br>representation<br>of the vote<br>count values. |
| 9 | `voteCount` | Checked for missing<br>values and filled any<br>with 0 | 0 cells modified | API output can occasionally omit<br>the vote field for zero-interaction<br>reviews; checked, no missing values<br>were present in this scrape. | Nothing — no<br>values were<br>filled, since<br>none were<br>missing. |
| 10 | `updated` | Parsed ISO-8601 string<br>to UTC Timestamp (`at`) | 4,092 cells<br>converted | Raw feed times are ingested as<br>static text strings. Chronological<br>tracking and time-series aggregation<br>require datetime-aware objects. | The original<br>timezone-offset<br>string syntax<br>and text<br>structure. |
| 11 | `im:version` | Converted to category<br>dtype<br>(`reviewCreatedVersion`) | 4,092 cells<br>converted | Over 4,000 reviews map to a small<br>set of unique Discord versions.<br>Categorical dtype reduces memory<br>footprint and optimizes group-by<br>query execution. | The dynamic<br>object<br>representation,<br>restricting the<br>set of levels to<br>observed<br>categories. |
| 12 | `im:rating` | Dropped raw column<br>after transformation | 4,092 cells<br>dropped | Keeping the duplicated raw-typed<br>text equivalent of the engineered<br>`score` column clutters working<br>memory and degrades tabular<br>display readability. | Instant workspace<br>access to the<br>uncleaned<br>raw-typed<br>column series. |
| 13 | `im:version` | Dropped raw column<br>after transformation | 4,092 cells<br>dropped | Keeping the duplicated raw-typed<br>text equivalent of the engineered<br>`reviewCreatedVersion` column<br>clutters working memory and<br>degrades tabular display readability. | Instant workspace<br>access to the<br>uncleaned<br>raw-typed<br>column series. |
| 14 | `updated` | Dropped raw column<br>after transformation | 4,092 cells<br>dropped | Keeping the duplicated raw-typed<br>text equivalent of the engineered<br>`at` column clutters working memory<br>and degrades tabular display<br>readability. | Instant workspace<br>access to the<br>uncleaned<br>raw-typed<br>column series. |
| 15 | `im:voteCount` | Dropped raw column<br>after transformation | 4,092 cells<br>dropped | Keeping the duplicated raw-typed<br>text equivalent of the engineered<br>`voteCount` column clutters working<br>memory and degrades tabular<br>display readability. | Instant workspace<br>access to the<br>uncleaned<br>raw-typed<br>column series. |
| 16 | `author` | Renamed column to<br>`userName` | 1 column header<br>renamed; 0 cell<br>values affected | The raw attribute name 'author'<br>was replaced with 'userName' to<br>comply with standardized app-store<br>schema mappings across Google Play<br>and iOS storefronts. | The original field<br>identifier label<br>'author' in the<br>working<br>dataframe. |
| 17 | DataFrame | Checked for<br>exact-duplicate rows | 0 rows dropped | Identical API packets are sometimes<br>re-fetched due to network retries,<br>creating artificial duplicates;<br>checked, none found — every logged<br>review is a unique transaction. | Nothing — 0 rows<br>met the duplicate<br>criteria; all 4,092<br>unique<br>transactions were<br>preserved. |

## Cleaning Decisions and Alternatives Considered

Numeric fields were coerced to NaN rather than dropped on failure, preserving every row for inspection. Missing voteCount was filled with 0 rather than left as NaN, since the API's missing-vote-count case means "zero votes," not "unknown." im:version was cast to category instead of left as a string for memory/query efficiency, since it holds a small repeating set of values. Raw columns were dropped after their cleaned replacements were created rather than kept side-by-side, since data/raw/ already preserves an undo path. Cells containing Mojibake are believed to be false positives as their text contains correctly witten foreign writing.

Deduplication used full-row matching instead of id-only matching; id-only would catch partial re-fetch mismatches, but none were found, so the simpler check was sufficient. Whitespace and mojibake fixes were applied in place rather than flagged for manual review, since zero cells needed correction in either case.

# 4. Row and column accounting

### Row Reconciliation
| Row Metric | Count |
| :--- | :--- |
| **Rows in raw file (`df_raw`)** | **4,092** |
| Rows removed as exact duplicates | - 0 |
| Rows removed as near-duplicates | - 0 |
| Rows removed for other reasons | - 0 |
| **Rows in cleaned file (`df`)** | **4,092** |

*Validation: 4,092 - 0 - 0 - 0 = 4,092 (Reconciled)*

---

### Column Reconciliation
| Column Metric | Count | Details |
| :--- | :--- | :--- |
| **Columns in raw file** | **10** | `['country', 'storefront', 'id', 'updated', 'title', 'content', 'im:rating', 'im:version', 'author', 'im:voteCount']` |
| Columns dropped | **- 4** | Dropped raw redundant inputs after casting: `im:rating`, `im:version`, `updated`, `im:voteCount` |
| Columns added / renamed | **+ 4** | Transformed/renamed columns added: `score`, `voteCount`, `at`, `reviewCreatedVersion` (and renamed `author` to `userName`) |
| **Columns in cleaned file** | **10** | `['country', 'storefront', 'id', 'title', 'content', 'userName', 'score', 'voteCount', 'at', 'reviewCreatedVersion']` |

*Validation: 10 (Raw) - 4 (Dropped) + 4 (Added) = 10 (Cleaned) (Reconciled)*

## 5. The Provenance Brief

This dataset contains 4,092 public reviews written by Discord users across nine international Apple App Store storefronts including: The United States, Brazil, Russia, India, United Kingdom, Germany, Canada, Australia and New Zealand. Users submitted these ratings and written feedback to express their experiences, report issues, or suggest improvements to the app developers.

Apple collects and hosts this feedback to help other consumers make informed download decisions and to give developers diagnostic insights. The data contains the review dates, titles, full message content, star ratings from 1 to 5, version of the app used, and user upvotes showing how helpful others found the review.

**What it supports:** This data is excellent for tracking general user sentiment over time, spotting common software bugs in specific app updates, or identifying major features that users either love or dislike.

**What it cannot support:** This data set contains no data or information that came from within the app itself. It does not represent all Discord users, as only a small fraction of people write reviews. It cannot tell us how often people use the app, nor does it contain private account data or information from Android or desktop users.

#